In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display


conn = sqlite3.connect("vehicle_rental.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS vehicles (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    vehicle_name TEXT NOT NULL,
    vehicle_type TEXT,
    number TEXT UNIQUE,
    rent_per_day REAL,
    status TEXT DEFAULT 'Available'
)
""")


cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    phone TEXT
)
""")


cursor.execute("""
CREATE TABLE IF NOT EXISTS rentals (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_name TEXT,
    vehicle_number TEXT,
    days INTEGER,
    total_amount REAL,
    status TEXT DEFAULT 'Rented'
)
""")

conn.commit()



def add_vehicle():
    print("\n--- ADD VEHICLE ---")

    name = input("Vehicle Name: ")
    vehicle_type = input("Vehicle Type (Car/Bike): ")
    number = input("Vehicle Number: ")
    rent = float(input("Rent Per Day: "))

    try:
        cursor.execute("""
        INSERT INTO vehicles
        (vehicle_name, vehicle_type, number, rent_per_day)
        VALUES (?, ?, ?, ?)
        """, (name, vehicle_type, number, rent))

        conn.commit()
        print("\nVehicle added successfully!")

    except sqlite3.IntegrityError:
        print("\nVehicle number already exists!")



def view_vehicles():
    print("\n--- VEHICLE DETAILS ---")

    data = pd.read_sql_query(
        "SELECT * FROM vehicles",
        conn
    )

    if data.empty:
        print("No vehicles available.")
    else:
        display(data)



def add_customer():
    print("\n--- ADD CUSTOMER ---")

    name = input("Customer Name: ")
    phone = input("Phone Number: ")

    cursor.execute("""
    INSERT INTO customers (name, phone)
    VALUES (?, ?)
    """, (name, phone))

    conn.commit()

    print("\nCustomer registered successfully!")



def view_customers():
    print("\n--- CUSTOMER DETAILS ---")

    data = pd.read_sql_query(
        "SELECT * FROM customers",
        conn
    )

    if data.empty:
        print("No customers found.")
    else:
        display(data)


def rent_vehicle():
    print("\n--- RENT VEHICLE ---")

    customer = input("Customer Name: ")
    vehicle_number = input("Vehicle Number: ")
    days = int(input("Number of Days: "))

    cursor.execute("""
    SELECT rent_per_day, status
    FROM vehicles
    WHERE number = ?
    """, (vehicle_number,))

    vehicle = cursor.fetchone()

    if vehicle is None:
        print("\nVehicle not found!")
        return

    rent_per_day, status = vehicle

    if status != "Available":
        print("\nVehicle is already rented!")
        return

    total = rent_per_day * days

    cursor.execute("""
    INSERT INTO rentals
    (customer_name, vehicle_number, days, total_amount)
    VALUES (?, ?, ?, ?)
    """, (customer, vehicle_number, days, total))

    cursor.execute("""
    UPDATE vehicles
    SET status = 'Rented'
    WHERE number = ?
    """, (vehicle_number,))

    conn.commit()

    print("\nVehicle rented successfully!")
    print("Total Amount: ₹", total)


def return_vehicle():
    print("\n--- RETURN VEHICLE ---")

    vehicle_number = input("Vehicle Number: ")

    cursor.execute("""
    SELECT id
    FROM rentals
    WHERE vehicle_number = ?
    AND status = 'Rented'
    """, (vehicle_number,))

    rental = cursor.fetchone()

    if rental is None:
        print("\nNo active rental found!")
        return

    cursor.execute("""
    UPDATE rentals
    SET status = 'Returned'
    WHERE id = ?
    """, (rental[0],))

    cursor.execute("""
    UPDATE vehicles
    SET status = 'Available'
    WHERE number = ?
    """, (vehicle_number,))

    conn.commit()

    print("\nVehicle returned successfully!")



def view_rentals():
    print("\n--- RENTAL RECORDS ---")

    data = pd.read_sql_query(
        "SELECT * FROM rentals",
        conn
    )

    if data.empty:
        print("No rental records found.")
    else:
        display(data)



while True:

    print("\n==========================================")
    print("     CLOUD-BASED VEHICLE RENTAL SYSTEM")
    print("==========================================")
    print("1. Add Vehicle")
    print("2. View Vehicles")
    print("3. Add Customer")
    print("4. View Customers")
    print("5. Rent Vehicle")
    print("6. Return Vehicle")
    print("7. View Rental Records")
    print("8. Exit")
    print("==========================================")

    choice = input("Enter your choice: ")

    if choice == "1":
        add_vehicle()

    elif choice == "2":
        view_vehicles()

    elif choice == "3":
        add_customer()

    elif choice == "4":
        view_customers()

    elif choice == "5":
        rent_vehicle()

    elif choice == "6":
        return_vehicle()

    elif choice == "7":
        view_rentals()

    elif choice == "8":
        print("\nThank you for using the Vehicle Rental System!")
        break

    else:
        print("\nInvalid choice. Please try again.")


conn.close()


     CLOUD-BASED VEHICLE RENTAL SYSTEM
1. Add Vehicle
2. View Vehicles
3. Add Customer
4. View Customers
5. Rent Vehicle
6. Return Vehicle
7. View Rental Records
8. Exit
Enter your choice: 1

--- ADD VEHICLE ---
Vehicle Name: toyota
Vehicle Type (Car/Bike): car
Vehicle Number: TN01AB1234
Rent Per Day: 1500

Vehicle added successfully!

     CLOUD-BASED VEHICLE RENTAL SYSTEM
1. Add Vehicle
2. View Vehicles
3. Add Customer
4. View Customers
5. Rent Vehicle
6. Return Vehicle
7. View Rental Records
8. Exit
